In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

from sklearn.model_selection import StratifiedKFold

In [5]:
project_dir = "../../../../../../../../s/project/gene_embedding/"



In [6]:
emogi_pred = pd.read_csv(project_dir + 'input_data/cancer_eval/emogi_predictions2.tsv', sep = '\t')

In [7]:
# read emogi train test sets
train_test = pd.read_csv(project_dir + 'input_data/cancer_eval/emogi_train_test.tsv', sep = '\t')
train_test



,gene_id,mask,y,dataset
0,ENSG00000232957,True,False,train
1,ENSG00000131966,True,False,train
2,ENSG00000133703,True,True,train
3,ENSG00000136240,True,False,train
4,ENSG00000137171,True,False,train
...,...,...,...,...
2953,ENSG00000256045,True,False,test
2954,ENSG00000151967,True,False,test
2955,ENSG00000256222,True,False,test
2956,ENSG00000269028,True,False,test


In [8]:
emogi_pred

,gene_id,label,pred
0,ENSG00000121879,True,5.293305
1,ENSG00000149311,True,5.293305
2,ENSG00000167548,True,5.293305
3,ENSG00000100393,True,5.293305
4,ENSG00000133703,True,5.293305
...,...,...,...
13174,ENSG00000169344,False,-5.293305
13175,ENSG00000167080,False,-5.293305
13176,ENSG00000237388,False,-5.293305
13177,ENSG00000254834,False,-5.293305


In [9]:
emogi_pred.gene_id.unique().shape

(13134,)

In [10]:
train_test = train_test.merge(emogi_pred, left_on = 'gene_id', right_on = 'gene_id', how = 'left').drop_duplicates()

In [11]:
folder = project_dir + 'embedding/combination/'
emb = pd.read_csv(folder + 'dtf_gtex_depMap_portT5_lunar-snowflake-239_3787637_embedding.tsv',
                   sep = '\t').set_index('gene_id')

In [12]:
emb = pd.read_csv(folder + 'combined_STRING.tsv',
                   sep = '\t').set_index('gene_id')

In [13]:
emb = pd.read_csv(folder + 'combined_STRING_EXP.tsv',
                   sep = '\t').set_index('gene_id')

In [14]:
df = train_test.merge(emb, on = 'gene_id')

In [15]:
df.query("dataset == 'train'")

,gene_id,mask,y,dataset,label,pred,node2vec_STRING_EXP_0,node2vec_STRING_EXP_1,node2vec_STRING_EXP_2,node2vec_STRING_EXP_3,...,verse_STRING_EXP_14,verse_STRING_EXP_15,verse_STRING_EXP_16,verse_STRING_EXP_17,verse_STRING_EXP_18,verse_STRING_EXP_19,verse_STRING_EXP_20,verse_STRING_EXP_21,verse_STRING_EXP_22,verse_STRING_EXP_23
0,ENSG00000131966,True,False,train,False,1.764483,1.065214,-0.075364,0.931517,-1.025243,...,-0.161952,-0.021930,0.174874,1.637834,-1.917963,0.759693,-0.590299,-1.193346,-0.329025,-0.997314
1,ENSG00000133703,True,True,train,True,5.293305,0.827782,-0.541780,0.471949,-0.426883,...,-0.505000,-0.389195,0.862868,3.074623,-2.887886,-0.571906,0.600656,-1.050709,0.452098,-0.582167
2,ENSG00000136240,True,False,train,False,-0.739284,0.861886,-0.163812,0.854272,-0.655536,...,-0.075061,0.451240,-0.271276,1.743656,-3.042485,0.885536,-0.039128,-0.738944,-0.615198,-0.645587
3,ENSG00000137171,True,False,train,False,1.239326,1.138923,0.020794,1.109097,-1.016797,...,-0.099517,0.832886,0.211886,1.871126,-1.193295,0.572535,-0.618961,-1.128620,1.368252,-1.383505
4,ENSG00000086598,True,False,train,False,-0.579617,0.925137,-0.769476,0.565440,-0.244538,...,-0.007578,0.048079,0.578325,2.297482,-2.228118,1.622870,0.317728,-1.238015,0.543348,-1.392682
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1975,ENSG00000002933,True,False,train,False,-5.288258,1.147264,-0.062711,0.451589,-0.649124,...,-0.802671,-1.061808,0.841813,1.282028,-2.080097,-0.643065,0.772291,-0.292963,0.974520,-2.037893
1976,ENSG00000215910,True,False,train,False,-5.293305,0.696704,-0.129353,0.457964,-0.579992,...,0.034564,-0.110039,-1.179528,1.156503,-3.031286,0.399005,-0.831258,-0.521551,0.733658,0.318496
1977,ENSG00000182544,True,False,train,False,-5.293305,0.633908,-0.163171,0.502114,-0.769476,...,-0.613361,0.439890,0.306199,1.204760,-2.243881,0.304900,2.326099,-1.861138,-0.342104,-1.433930
1978,ENSG00000092931,True,False,train,False,-5.293305,0.717116,-0.205804,0.523099,-0.431104,...,-0.925421,1.371548,0.160410,-0.225177,-2.220284,0.491525,0.049502,-0.761783,1.457828,-1.913063


In [16]:
#kf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 1234)

pred = []

for method in ["score_only", "and_emb", "emb_only"]:


        ## copy p val only in case features is pval
        if method == 'score_only':
            score = df.query("dataset == 'test'")['pred']

        ## otherwise train gradient boosting tree.
        else:
            #boost = HistGradientBoostingClassifier(learning_rate = 0.014 , max_iter = max_iter, max_depth = 4)

            boost = HistGradientBoostingClassifier()

            features_to_drop = ["gene_id", "mask", "y", "dataset", "label"]
            if method == 'emb_only':
                features_to_drop = features_to_drop + ["pred"]

            boost.fit(df.query("dataset == 'train'").drop(features_to_drop, axis=1), df.query("dataset == 'train'").y)
            score = boost.predict_proba(df.query("dataset == 'test'").drop(features_to_drop, axis=1))[:,1]

        pred.append(
            pd.DataFrame({
                "gene_id": df.query("dataset == 'test'")["gene_id"],
                "pred": score,
                "label": df.query("dataset == 'test'")["label"],
                "original_score": df.query("dataset == 'test'")["pred"],
                "method": method, 
            })
        )


pred = pd.concat(pred)
# pred.to_csv(snakemake.output['pred'], sep = '\t', index = False)

In [17]:
pred

,gene_id,pred,label,original_score,method
1980,ENSG00000135720,-0.494110,False,-0.494110,score_only
1981,ENSG00000075785,-2.117152,False,-2.117152,score_only
1982,ENSG00000170759,0.918956,True,0.918956,score_only
1983,ENSG00000138069,-1.273835,False,-1.273835,score_only
1984,ENSG00000053501,-2.084184,False,-2.084184,score_only
...,...,...,...,...,...
2718,ENSG00000256045,0.000418,False,-5.293305,emb_only
2719,ENSG00000151967,0.033571,False,-5.293298,emb_only
2720,ENSG00000256222,0.000646,False,-5.293299,emb_only
2721,ENSG00000269028,0.000498,False,-2.476709,emb_only


In [18]:
pred.to_csv(project_dir + 'processed_data/emogi_pred/emogi_test_set.tsv', sep = '\t')

In [19]:
pred.to_csv(project_dir + 'processed_data/emogi_pred/emogi_STRING_test_set.tsv', sep = '\t')

In [20]:
pred.to_csv(project_dir + 'processed_data/emogi_pred/emogi_STRING_EXP_test_set.tsv', sep = '\t')